# 01 — Construire les tables du Lakehouse Fabric

**Rôle : dernière étape du pipeline de l'équipe**, après `citiz` 01 et 03, `regional-data` 07 et les carreaux de `population/`. **Statut : préparatoire**, pas encore exécuté dans Fabric.

**Objectif** : transformer les données Citiz explorées, les contours administratifs, les indicateurs OSM de `regional-data/`, la population de `population/` et les paramètres de coût en **tables propres, typées et contrôlées**, prêtes pour un Lakehouse Microsoft Fabric.

**Seules les colonnes jugées exploitables dans le notebook `citiz` 02 sont reprises** (voir sa section 6). Les tables de disponibilité sont des **annexes** : conservées pour l'historique, pas une mesure d'usage.

**Lakehouse** : espace de stockage Fabric avec deux zones :
- `Files/` : des fichiers (CSV, Parquet, GeoJSON) ;
- `Tables/` : des tables **Delta**, interrogeables en SQL et lisibles par Power BI.

**Organisation proposée**

```
Files/raw/gbfs/        CSV bruts du notebook `citiz` 01 (jamais modifiés)
Files/raw/boundaries/  contours régions / départements
Files/raw/citiz/       citiz_stations_geo.csv du notebook `citiz` 03 (département de chaque station)
Files/raw/regional/    citiz_aura_final.csv, table finale de regional-data/ (indicateurs OSM)
Files/raw/population/  carreaux_200m_aura.csv, carreaux INSEE Filosofi 2021 de population/
Files/clean/           une copie Parquet de chaque table
Tables/                tables Delta
```

**Deux environnements**
- **Fabric** (`spark` existe) : lit `Files/raw/`, écrit `Files/clean/` puis les tables Delta.
- **Local** : écrit un miroir de cette arborescence dans `lakehouse/data/`, à déposer tel quel dans le Lakehouse.

**Règles appliquées à toutes les tables**
- noms de tables et de colonnes en anglais, minuscules, sans espace ni accent ;
- identifiants en texte ;
- dates en UTC à la microseconde (Spark ne lit pas les nanosecondes de pandas) ;
- géométries en texte WKT + `lat` / `lon` : Delta n'a pas de type géométrique ;
- une clé primaire par table, vérifiée avant écriture ;
- colonne `_loaded_at` : date du chargement ;
- une donnée manquante reste vide, jamais remplacée par zéro.

In [1]:
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path
import requests
import pandas as pd
import geopandas as gpd

try:
    spark  # noqa: F821  (defined automatically in Fabric)
    IN_FABRIC = True
except NameError:
    IN_FABRIC = False

if IN_FABRIC:
    lakehouse_directory = Path("/lakehouse/default")
else:
    from pathlib import Path

    current_directory = Path.cwd().resolve()
    project_directory = None
    for directory in [current_directory, *current_directory.parents]:
        if (directory / "citiz" / "AGENTS.md").exists():
            project_directory = directory
            break
    if project_directory is None:
        raise FileNotFoundError("Open this notebook from the repository directory.")

    module_directory = project_directory / "citiz"
    gbfs_directory = module_directory / "data" / "raw" / "gbfs"
    boundaries_directory = module_directory / "data" / "raw" / "boundaries"
    processed_directory = module_directory / "data" / "processed"
    for directory in [gbfs_directory, boundaries_directory, processed_directory]:
        directory.mkdir(parents=True, exist_ok=True)
    lakehouse_directory = project_directory / "lakehouse" / "data"

raw_gbfs_directory = lakehouse_directory / "Files" / "raw" / "gbfs"
raw_boundaries_directory = lakehouse_directory / "Files" / "raw" / "boundaries"
raw_citiz_directory = lakehouse_directory / "Files" / "raw" / "citiz"
raw_regional_directory = lakehouse_directory / "Files" / "raw" / "regional"
raw_population_directory = lakehouse_directory / "Files" / "raw" / "population"
clean_directory = lakehouse_directory / "Files" / "clean"
for directory in [raw_gbfs_directory, raw_boundaries_directory, raw_citiz_directory, raw_regional_directory,
                  raw_population_directory, clean_directory]:
    directory.mkdir(parents=True, exist_ok=True)

LOADED_AT = pd.Timestamp(datetime.now(timezone.utc))
print("Fabric" if IN_FABRIC else "Local", "->", lakehouse_directory if IN_FABRIC else lakehouse_directory.relative_to(project_directory))

Local -> lakehouse/data


## 1. Rassembler les fichiers bruts

- **En local** : copie des CSV de `citiz/data/raw/gbfs/`, des contours de `citiz/data/raw/boundaries/`, de `citiz/data/processed/citiz_stations_geo.csv` (notebook `citiz` 03), de `regional-data/data/release/citiz_aura_final.csv` et de `population/reference/filosofi_2021_aura/carreaux_200m_aura.csv`.
- **Dans Fabric** : déposer d'abord ces fichiers dans `Files/raw/gbfs/`, `Files/raw/citiz/`, `Files/raw/regional/` et `Files/raw/population/` (Lakehouse > Files > Upload). Les contours sont téléchargés s'ils manquent.

In [2]:
GBFS_FILES = ["citiz_flux.csv", "citiz_stations.csv", "citiz_station_status.csv", "citiz_vehicle_types.csv"]
STATION_GEO_FILE = raw_citiz_directory / "citiz_stations_geo.csv"
REGIONAL_FILE = raw_regional_directory / "citiz_aura_final.csv"
POPULATION_FILE = raw_population_directory / "carreaux_200m_aura.csv"
BOUNDARY_URLS = {
    "regions": "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/regions-version-simplifiee.geojson",
    "departements": "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-version-simplifiee.geojson",
}

if not IN_FABRIC:
    for file_name in GBFS_FILES:
        shutil.copy(gbfs_directory / file_name, raw_gbfs_directory / file_name)
    for name in BOUNDARY_URLS:
        local_copy = boundaries_directory / f"{name}.geojson"
        if local_copy.exists():
            shutil.copy(local_copy, raw_boundaries_directory / local_copy.name)
    shutil.copy(processed_directory / "citiz_stations_geo.csv", STATION_GEO_FILE)
    shutil.copy(project_directory / "regional-data" / "data" / "release" / "citiz_aura_final.csv", REGIONAL_FILE)
    shutil.copy(project_directory / "population" / "reference" / "filosofi_2021_aura" / "carreaux_200m_aura.csv",
                POPULATION_FILE)

missing = [file_name for file_name in GBFS_FILES if not (raw_gbfs_directory / file_name).exists()]
assert not missing, f"Upload to Files/raw/gbfs/: {missing}"
for path in [STATION_GEO_FILE, REGIONAL_FILE, POPULATION_FILE]:
    assert path.exists(), f"Upload {path.name} to {path.parent}"

for name, url in BOUNDARY_URLS.items():
    path = raw_boundaries_directory / f"{name}.geojson"
    if not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
print(sorted(path.name for path in raw_gbfs_directory.iterdir()))

['citiz_flux.csv', 'citiz_station_status.csv', 'citiz_stations.csv', 'citiz_vehicle_types.csv']


## 2. Outils communs

- `typed` : garde uniquement les colonnes du schéma, dans l'ordre, avec le bon type. Une colonne absente de la source est créée vide, pour que le schéma reste stable.
- `save_table` : vérifie la clé primaire, écrit le Parquet dans `Files/clean/`, puis la table Delta dans Fabric.

In [3]:
def typed(df, schema):
    """Project df on schema = {column: type}; types: string, Int64, float64, boolean, datetime."""
    result = pd.DataFrame(index=df.index)
    for column, dtype in schema.items():
        values = df[column] if column in df else pd.Series(pd.NA, index=df.index, dtype="object")
        if dtype == "datetime":
            result[column] = pd.to_datetime(values, utc=True, errors="coerce", format="ISO8601")
        else:
            result[column] = values.astype(dtype)
    return result.reset_index(drop=True)


def save_table(df, name, primary_key):
    """Check the primary key, then write Files/clean/<name>.parquet (and the Delta table in Fabric)."""
    duplicates = df.duplicated(primary_key).sum()
    assert duplicates == 0, f"{name}: {duplicates} duplicates on {primary_key}"
    assert df[primary_key].notna().all().all(), f"{name}: incomplete key {primary_key}"
    df = df.assign(_loaded_at=LOADED_AT)
    df.to_parquet(clean_directory / f"{name}.parquet", index=False,
                  coerce_timestamps="us", allow_truncated_timestamps=True)
    if IN_FABRIC:
        (spark.read.parquet(f"Files/clean/{name}.parquet")  # noqa: F821
              .write.format("delta").mode("overwrite")
              .option("overwriteSchema", "true").saveAsTable(name))
    print(f"{name:<30} {len(df):>6} rows  key={primary_key}")
    return df


ID_COLUMNS = {"station_id": "string", "vehicle_type_id": "string", "post_code": "string"}


def read_raw(file_name):
    return pd.read_csv(raw_gbfs_directory / file_name, dtype=ID_COLUMNS)


tables = {}

## 3. Référentiel géographique : `geo_region`, `geo_department`

Colonnes source renommées : `nom` → `name`. Le département reçoit le code de sa région (jointure spatiale sur un point intérieur), pour filtrer « Rhône » ou « Auvergne-Rhône-Alpes » sans recalcul géographique.

In [4]:
regions = gpd.read_file(raw_boundaries_directory / "regions.geojson")
departments = gpd.read_file(raw_boundaries_directory / "departements.geojson")

inner_points = departments.copy()
inner_points["geometry"] = departments.representative_point()
department_region = gpd.sjoin(inner_points, regions.rename(columns={"code": "region_code"})[["region_code", "geometry"]],
                              how="left", predicate="within")
departments["region_code"] = department_region["region_code"].values


def boundary_table(gdf, schema):
    df = pd.DataFrame(gdf.drop(columns="geometry")).rename(columns={"nom": "name"})
    df["geometry_wkt"] = gdf.geometry.to_wkt()
    centroids = gdf.to_crs("EPSG:2154").geometry.centroid.to_crs("EPSG:4326")
    df["centroid_lat"] = centroids.y
    df["centroid_lon"] = centroids.x
    return typed(df, schema)


BOUNDARY_SCHEMA = {"code": "string", "name": "string", "centroid_lat": "float64", "centroid_lon": "float64",
                   "geometry_wkt": "string"}
tables["geo_region"] = save_table(boundary_table(regions, BOUNDARY_SCHEMA), "geo_region", ["code"])
tables["geo_department"] = save_table(boundary_table(departments, {**BOUNDARY_SCHEMA, "region_code": "string"}),
                                      "geo_department", ["code"])

geo_region                         13 rows  key=['code']
geo_department                     96 rows  key=['code']


## 4. Citiz : réseaux, stations, types de véhicules

- `citiz_network` : un flux régional par ligne, clé `source`, avec un nom court (`network_name`).
- `citiz_station` : clé `station_uid`. Le département vient de `citiz_stations_geo.csv` (notebook `citiz` 03) : le rattachement spatial n'est fait qu'une fois, et le Lakehouse a les mêmes départements que la référence d'équipe. La région s'en déduit par `geo_department`. L'état instantané (`num_vehicles_available`, `is_renting`) est retiré : il appartient à `citiz_station_status`. Les colonnes vides partout (`short_name`, `post_code`, `city`) ne sont pas reprises. `is_internal` signale les 3 pseudo-stations internes (stockage, démonstration, livraison) repérées par leur nom dans le notebook `citiz` 02 : elles restent dans la table, à filtrer avant toute analyse.
- `citiz_vehicle_type` : clé `vehicle_type_uid` = `source:vehicle_type_id`. `max_range_meters` (0 pour 96 % des modèles) et `rider_capacity` (vide) ne sont pas reprises.

In [5]:
feeds = read_raw("citiz_flux.csv")
feeds["network_name"] = feeds.dataset_title.str.replace("Autopartage Citiz ", "", regex=False)
tables["citiz_network"] = save_table(typed(feeds, {
    "source": "string", "network_name": "string", "dataset_title": "string", "publisher": "string",
    "gbfs_url": "string", "gbfs_version": "string", "feeds": "string", "licence": "string",
    "page_url": "string", "dataset_id": "string", "is_available": "boolean",
}), "citiz_network", ["source"])

citiz_network                      15 rows  key=['source']


In [6]:
stations = read_raw("citiz_stations.csv")
stations["station_uid"] = stations.source + ":" + stations.station_id
station_points = gpd.GeoDataFrame(stations, geometry=gpd.points_from_xy(stations.lon, stations.lat), crs="EPSG:4326")
station_geo = pd.read_csv(STATION_GEO_FILE, usecols=["station_uid", "dep_code"], dtype="string")
assert set(station_geo.station_uid) == set(stations.station_uid), "Re-run notebook 03 after notebook 01."
station_points = station_points.merge(station_geo, on="station_uid", how="left", validate="one_to_one")
station_points["region_code"] = station_points.dep_code.map(departments.set_index("code").region_code)
print(f"Stations without department: {station_points.dep_code.isna().sum()}")
station_points["geometry_wkt"] = station_points.geometry.to_wkt()
INTERNAL_NAME_PATTERN = r"^ZZ_|hors.circuit|DEMOBOX|NE PAS RESERVER"
station_points["is_internal"] = station_points.name.str.contains(INTERNAL_NAME_PATTERN, case=False, regex=True)
print(f"Internal pseudo-stations: {station_points.is_internal.sum()}")

tables["citiz_station"] = save_table(typed(station_points, {
    "station_uid": "string", "source": "string", "station_id": "string", "name": "string",
    "address": "string",
    "lat": "float64", "lon": "float64", "geometry_wkt": "string",
    "dep_code": "string", "region_code": "string",
    "capacity": "Int64", "is_virtual_station": "boolean", "is_internal": "boolean", "sources": "string",
    "extracted_at": "datetime",
}), "citiz_station", ["station_uid"])

Stations without department: 0
Internal pseudo-stations: 3


citiz_station                    1615 rows  key=['station_uid']


In [7]:
vehicle_types = read_raw("citiz_vehicle_types.csv")
vehicle_types["vehicle_type_uid"] = vehicle_types.source + ":" + vehicle_types.vehicle_type_id
tables["citiz_vehicle_type"] = save_table(typed(vehicle_types, {
    "vehicle_type_uid": "string", "source": "string", "vehicle_type_id": "string", "name": "string",
    "make": "string", "model": "string", "form_factor": "string", "propulsion_type": "string",
}), "citiz_vehicle_type", ["vehicle_type_uid"])

citiz_vehicle_type                337 rows  key=['vehicle_type_uid']


## 5. Indicateurs OSM autour des stations : `station_feature_aura`

Table finale de `regional-data/` (`citiz_aura_final.csv`) : équipements, stationnements vélo, parkings, sites de recharge et de tourisme accessibles à pied par le réseau piéton OSM, pour les stations d'Auvergne-Rhône-Alpes. Clé `station_uid`, liée à `citiz_station`.

Seuls les indicateurs calculés sont repris :
- **non repris, déjà dans `citiz_station`** : nom, position, département, région, réseau, capacité ;
- **non repris, non exploitables** : `city` (vide), `is_renting` (constante), `num_vehicles_available` (instantané périmé), `extracted_at`, `station_reference_match` ;
- **non repris, pas encore calculés** : `irve_count_500m`, `transit_departures_peak_hour`, `pedestrian_network_distance_m`, `accidents_count_500m` (vides) et leurs colonnes `_status`, constantes ;
- **non repris, constants** : `walking_network_status`, `osm_count_status`, `osm_extraction_has_geometry_issues` (voir le README de `regional-data/`).

`public_parking_spaces_300m` reste vide quand l'accès ou la capacité d'un parking est inconnu : ce n'est pas un zéro.

In [8]:
COUNT_COLUMNS = [
    "amenities_count_500m", "amenities_count_500m_straight_line", "amenities_unreachable_or_unsnapped_500m",
    "bike_parking_count_300m", "bike_parking_count_300m_straight_line", "bike_parking_unreachable_or_unsnapped_300m",
    "tourism_sites_count_1km", "tourism_sites_count_1km_straight_line", "tourism_unreachable_or_unsnapped_1km",
    "osm_charging_sites_count_500m", "osm_charging_sites_count_500m_straight_line",
    "charging_sites_unreachable_or_unsnapped_500m",
    "parking_sites_count_300m", "parking_sites_count_300m_straight_line", "parking_unreachable_or_unsnapped_300m",
    "public_parking_spaces_300m", "public_parking_spaces_known_300m", "public_parking_sites_300m",
    "parking_sites_unknown_access_300m", "public_parking_sites_unknown_capacity_300m",
]
FEATURE_SCHEMA = {
    "station_uid": "string",
    **{column: "Int64" for column in COUNT_COLUMNS},
    "public_parking_spaces_300m_status": "string",
    "walking_station_snap_distance_m": "float64",
    "radius_300m_inside_ign_region": "boolean", "radius_500m_inside_ign_region": "boolean",
    "radius_1000m_inside_ign_region": "boolean", "boundary_coverage_status": "string",
}

features = pd.read_csv(REGIONAL_FILE, encoding="utf-8-sig", dtype={"station_uid": "string"}, low_memory=False)
missing_columns = set(FEATURE_SCHEMA) - set(features.columns)
assert not missing_columns, f"Columns missing from {REGIONAL_FILE.name}: {sorted(missing_columns)}"
print(f"Columns not loaded: {len(features.columns) - len(FEATURE_SCHEMA)} of {len(features.columns)}")
tables["station_feature_aura"] = save_table(typed(features, FEATURE_SCHEMA), "station_feature_aura", ["station_uid"])

Columns not loaded: 23 of 50
station_feature_aura              612 rows  key=['station_uid']


## 6. Population : `population_grid_aura`

Carreaux de 200 m de l'INSEE (Filosofi 2021) préparés dans `population/` : une ligne par carreau, clé `idcar_200m`. Les valeurs sont celles du fichier INSEE, ajustées pour la confidentialité (décimales possibles) ; leurs définitions sont dans `population/DATA_DICTIONARY.md`.

Colonnes ajoutées :
- `geometry_wkt` : contour du carreau en WGS84, recalculé depuis le coin sud-ouest contenu dans `idcar_200m` (EPSG:3035). C'est la zone à colorer pour l'objectif 1 ; `lon` / `lat` (centre) viennent de `population/` ;
- `dep_code` : département, lu dans les codes commune de `lcog_geo`. Un carreau qui chevauche deux départements garde `dep_code` vide et `is_multi_department = true` : on ne choisit pas un département au hasard.

In [9]:
GRID_ID_PATTERN = r"CRS(?P<epsg>\d+)RES(?P<size>\d+)mN(?P<north>\d+)E(?P<east>\d+)"
POPULATION_COLUMNS = ["ind", "men", "men_pauv", "men_coll", "ind_snv", "ind_0_3", "ind_4_5", "ind_6_10", "ind_11_17",
                      "ind_18_24", "ind_25_39", "ind_40_54", "ind_55_64", "ind_65_79", "ind_80p"]

grid = pd.read_csv(POPULATION_FILE, dtype={"idcar_200m": "string", "lcog_geo": "string"})
corners = grid.idcar_200m.str.extract(GRID_ID_PATTERN)
assert corners.notna().all().all(), "Unexpected idcar_200m format"
assert corners.epsg.nunique() == 1 and corners["size"].nunique() == 1
east, north, size = corners.east.astype(int), corners.north.astype(int), int(corners["size"].iloc[0])
squares = gpd.GeoSeries.from_xy(east + size / 2, north + size / 2, crs=f"EPSG:{corners.epsg.iloc[0]}").buffer(
    size / 2, cap_style="square")
grid["geometry_wkt"] = squares.to_crs("EPSG:4326").to_wkt(rounding_precision=6).values


def department_codes(lcog_geo):
    """Departments of the municipalities listed in lcog_geo (3 characters overseas)."""
    return {code[:3] if code.startswith("97") else code[:2] for code in lcog_geo.split(",")}


departments_by_cell = grid.lcog_geo.map(department_codes)
grid["is_multi_department"] = departments_by_cell.str.len() > 1
grid["dep_code"] = departments_by_cell.map(lambda codes: next(iter(codes)) if len(codes) == 1 else None)
print(f"Cells: {len(grid)} | inhabitants: {grid.ind.sum():,.0f} | on two departments: {grid.is_multi_department.sum()}")

tables["population_grid_aura"] = save_table(typed(grid, {
    "idcar_200m": "string", "lon": "float64", "lat": "float64", "geometry_wkt": "string",
    "lcog_geo": "string", "dep_code": "string", "is_multi_department": "boolean", "i_est_200": "Int64",
    **{column: "float64" for column in POPULATION_COLUMNS},
}), "population_grid_aura", ["idcar_200m"])

Cells: 312143 | inhabitants: 7,703,556 | on two departments: 2140


population_grid_aura           312143 rows  key=['idcar_200m']


## 7. Tables annexes : historique de disponibilité

- `citiz_station_status` : une ligne par station et par instantané (clé `station_uid` + `snapshot_ts`).
- `citiz_station_status_vehicle` : la colonne JSON `vehicle_types_available` dépliée, une ligne par type de véhicule disponible.

Rappel du notebook `citiz` 02 : `last_reported` est souvent vieux de plusieurs mois, le flux ne reflète donc pas les mouvements réels des voitures. Ces tables sont **annexes** : conservées pour accumuler l'historique, pas pour mesurer la fréquentation.

- `status_age_days` : jours entre l'instantané et `last_reported` ;
- `is_stale` : statut vieux de plus de 30 jours.

Colonnes non reprises : `num_vehicles_disabled` (vide), `num_docks_available` (toujours 0), `is_renting` (toujours vrai).

In [10]:
status = read_raw("citiz_station_status.csv")
status["station_uid"] = status.source + ":" + status.station_id
status = status.drop_duplicates(["station_uid", "snapshot_ts"], keep="last")
STALE_AFTER_DAYS = 30
age = (pd.to_datetime(status.snapshot_ts, utc=True, format="ISO8601")
       - pd.to_datetime(status.last_reported, utc=True, format="ISO8601"))
status["status_age_days"] = age.dt.days
status["is_stale"] = (status.status_age_days > STALE_AFTER_DAYS).where(status.status_age_days.notna())
print(f"Stale statuses (> {STALE_AFTER_DAYS} days): {status.is_stale.mean():.0%}")

tables["citiz_station_status"] = save_table(typed(status, {
    "station_uid": "string", "snapshot_ts": "datetime", "source": "string", "station_id": "string",
    "num_vehicles_available": "Int64", "is_installed": "boolean", "is_returning": "boolean",
    "last_reported": "datetime", "status_age_days": "Int64", "is_stale": "boolean",
}), "citiz_station_status", ["station_uid", "snapshot_ts"])

vehicle_rows = []
for row in status.itertuples():
    if isinstance(row.vehicle_types_available, str):
        for item in json.loads(row.vehicle_types_available):
            vehicle_rows.append({"station_uid": row.station_uid, "snapshot_ts": row.snapshot_ts,
                                 "vehicle_type_uid": f"{row.source}:{item['vehicle_type_id']}",
                                 "count": item.get("count")})
tables["citiz_station_status_vehicle"] = save_table(typed(pd.DataFrame(vehicle_rows), {
    "station_uid": "string", "snapshot_ts": "datetime", "vehicle_type_uid": "string", "count": "Int64",
}), "citiz_station_status_vehicle", ["station_uid", "snapshot_ts", "vehicle_type_uid"])

Stale statuses (> 30 days): 95%
citiz_station_status             6459 rows  key=['station_uid', 'snapshot_ts']
citiz_station_status_vehicle    10056 rows  key=['station_uid', 'snapshot_ts', 'vehicle_type_uid']


## 8. Paramètres de coût : `cost_parameter`

Paramètres du futur modèle de coût (objectif 3). `status` vaut :
- `sourced` : chiffre tiré d'un document public ;
- `derived` : calculé à partir de chiffres sourcés ;
- `assumption` : hypothèse à fixer (valeur vide pour l'instant, à demander à Citiz).

Modifier cette cellule puis relancer pour mettre la table à jour.

In [11]:
CONVENTION_URL = "https://genevoisfrancais.org/wp-content/uploads/CS2021_23.2-AUTOPARTAGE_Convention-cadreAnnexe.pdf"
WHITE_PAPER_URL = "https://www.citiz.coop/uploads/2026/02/2026-CITIZ-livre_blanc.pdf"

costs = pd.DataFrame([
    ("citiz", "telematics_box", 4000, "EUR excl. VAT / car", "sourced", CONVENTION_URL + " (art. 5.3)"),
    ("citiz", "fixed_operating_cost", 250, "EUR / month / car", "sourced", CONVENTION_URL + " (art. 6.4)"),
    ("citiz", "revenue_per_km", 0.19, "EUR / km", "sourced", CONVENTION_URL + " (art. 6.4)"),
    ("citiz", "min_cars_per_station", 2, "cars", "sourced", CONVENTION_URL + " (preamble)"),
    ("citiz", "break_even_km", 250 / 0.19, "km / month / car", "derived", "fixed_operating_cost / revenue_per_km"),
    ("citiz", "car_purchase_price", None, "EUR incl. VAT / car", "assumption", "to ask Citiz"),
    ("citiz", "station_setup_cost", None, "EUR / station", "assumption", WHITE_PAPER_URL + " (described as minimal)"),
    ("bus_stop", "stop_creation_cost", None, "EUR / stop", "assumption", "to research"),
    ("bike_sharing", "station_creation_cost", None, "EUR / station", "assumption", "to research"),
], columns=["facility", "parameter", "value", "unit", "status", "source"])

tables["cost_parameter"] = save_table(typed(costs, {
    "facility": "string", "parameter": "string", "value": "float64",
    "unit": "string", "status": "string", "source": "string",
}), "cost_parameter", ["facility", "parameter"])

cost_parameter                      9 rows  key=['facility', 'parameter']


## 9. Contrôles de qualité

Vérifie les liens entre tables (clés étrangères) : chaque statut pointe vers une station connue, chaque station vers un réseau et un département connus, etc.

In [12]:
def check_foreign_key(child, column, parent, parent_column):
    orphans = (~tables[child][column].dropna().isin(tables[parent][parent_column])).sum()
    label = "OK" if orphans == 0 else "KO"
    print(f"{label}  {child}.{column} -> {parent}.{parent_column}: {orphans} orphans")
    return orphans


orphan_total = sum([
    check_foreign_key("citiz_station", "source", "citiz_network", "source"),
    check_foreign_key("citiz_station", "dep_code", "geo_department", "code"),
    check_foreign_key("citiz_station", "region_code", "geo_region", "code"),
    check_foreign_key("station_feature_aura", "station_uid", "citiz_station", "station_uid"),
    check_foreign_key("population_grid_aura", "dep_code", "geo_department", "code"),
    check_foreign_key("citiz_station_status", "station_uid", "citiz_station", "station_uid"),
    check_foreign_key("citiz_station_status_vehicle", "vehicle_type_uid", "citiz_vehicle_type", "vehicle_type_uid"),
    check_foreign_key("citiz_vehicle_type", "source", "citiz_network", "source"),
    check_foreign_key("geo_department", "region_code", "geo_region", "code"),
])
assert orphan_total == 0

OK  citiz_station.source -> citiz_network.source: 0 orphans
OK  citiz_station.dep_code -> geo_department.code: 0 orphans
OK  citiz_station.region_code -> geo_region.code: 0 orphans
OK  station_feature_aura.station_uid -> citiz_station.station_uid: 0 orphans
OK  population_grid_aura.dep_code -> geo_department.code: 0 orphans
OK  citiz_station_status.station_uid -> citiz_station.station_uid: 0 orphans
OK  citiz_station_status_vehicle.vehicle_type_uid -> citiz_vehicle_type.vehicle_type_uid: 0 orphans
OK  citiz_vehicle_type.source -> citiz_network.source: 0 orphans
OK  geo_department.region_code -> geo_region.code: 0 orphans


In [13]:
ANNEX_TABLES = {"citiz_station_status", "citiz_station_status_vehicle"}
data_dictionary = pd.DataFrame([
    {"table": name, "role": "annex" if name in ANNEX_TABLES else "main", "column": column, "dtype": str(df[column].dtype),
     "missing": int(df[column].isna().sum()), "rows": len(df)}
    for name, df in tables.items() for column in df.columns
])
data_dictionary.to_csv(clean_directory / "_data_dictionary.csv", index=False)
data_dictionary.groupby(["role", "table"]).agg(rows=("rows", "first"), columns=("column", "size"))

rows  columns
role  table                                        
annex citiz_station_status            6459       11
      citiz_station_status_vehicle   10056        5
main  citiz_network                     15       12
      citiz_station                   1615       16
      citiz_vehicle_type               337        9
      cost_parameter                     9        7
      geo_department                    96        7
      geo_region                        13        6
      population_grid_aura          312143       24
      station_feature_aura             612       28

## 10. Vérification dans Fabric

Requête SQL de contrôle (Fabric uniquement).

In [14]:
if IN_FABRIC:
    display(spark.sql("""
        SELECT d.name AS department, COUNT(*) AS stations, SUM(s.capacity) AS capacity
        FROM citiz_station s JOIN geo_department d ON s.dep_code = d.code
        GROUP BY d.name ORDER BY stations DESC LIMIT 10
    """))  # noqa: F821
else:
    print(f"Local files ready in {clean_directory.relative_to(project_directory)}")
    print("Upload lakehouse/data/Files/ to the Lakehouse Files/, then run this notebook in Fabric.")

Local files ready in lakehouse/data/Files/clean
Upload lakehouse/data/Files/ to the Lakehouse Files/, then run this notebook in Fabric.
